## OCI Data Catalog API Demo

#### Prerequisites
1. create dynamic group matching rule - All {request.principal.type = 'aidataplatform',resource.compartment.id = '<compartment-ocid>'}
2. create policy - allow dynamic-group <dynamic-group> to manage data-catalog-family in compartment id <compartment-ocid>
3. setup requirements.txt with oci library configured (install in the compute server)
4. setup oci config and pem file in the compute server

In [72]:
import oci

COMPARTMENT_ID="ocid1.compartment.oc1..aaaaaaaay4b33s4cl7a3l3ehzklo3hsozhxcuuhil4nwnn5m76gjh4xypoua" 
config = oci.config.from_file("/Workspace/.oci/config")
oci.config.validate_config(config)
# Initialize service client with default config file
data_catalog_client = oci.data_catalog.DataCatalogClient(config)

In [73]:
# ------------------------------------------------------------------------
# List data catalogs
# ------------------------------------------------------------------------
list_catalogs_response = data_catalog_client.list_catalogs(
    compartment_id=COMPARTMENT_ID)

# Get the data from response
ids = [catalog.id for catalog in list_catalogs_response.data]
#print("Data catalog instance OCID: "+str(ids))

#print(list_catalogs_response.data)
for catalog in list_catalogs_response.data:
    print(f"Catalog key={catalog.id}\tCatalog display_name={catalog.display_name}")

# ------------------------------------------------------
# Display catalog attributes based on data catalog OCID
# ------------------------------------------------------
get_catalog_response = data_catalog_client.get_catalog(
    catalog_id=ids)

# Get the data from response
display_name = get_catalog_response.data.display_name
ocid=get_catalog_response.data.id
lifecycle_state=get_catalog_response.data.lifecycle_state
print("Name: "+display_name+" OCID: "+ocid+" lifecycle_state: "+lifecycle_state)

Catalog key=ocid1.datacatalog.oc1.us-chicago-1.amaaaaaa7ratczialrxpvt7va4hidvuh54lz2yboz44acxoc3btotzrwp63q	Catalog display_name=Pilot
Name: Pilot OCID: ocid1.datacatalog.oc1.us-chicago-1.amaaaaaa7ratczialrxpvt7va4hidvuh54lz2yboz44acxoc3btotzrwp63q lifecycle_state: ACTIVE


In [74]:
# ---------------------------------------------------------------
# List data catalog assets
# ----------------------------------------------------------------
list_data_assets_response = data_catalog_client.list_data_assets(
    catalog_id=ids)

# Get the data from response
#print(list_data_assets_response.data)
for asset in list_data_assets_response.data.items:
    print(f"Asset key={asset.key}\tAsset display_name={asset.display_name}") if asset.display_name == "Ellison_OS" else None

Asset key=672dd4d1-04d8-4839-9016-23654bb6c41e	Asset display_name=Ellison_OS


In [75]:
# --------------------------------------------------------------
# List data catalog entities for the data asset upto 10 entities
# --------------------------------------------------------------
ASSET_KEY="672dd4d1-04d8-4839-9016-23654bb6c41e"
list_entities_response = data_catalog_client.list_entities(
    catalog_id=ids,
    data_asset_key=ASSET_KEY)
list_entities_response = sorted(
    list_entities_response.data.items,
    key=lambda e: (e.display_name or "").lower()
)
# Get the data from response
for entity in list_entities_response[:2]: # Print ordered upto n rows
    print(f"Key={entity.key}\tEntity={entity.display_name}\tSource={entity.external_key}") if "json" in entity.display_name.lower() else None
for entity in list_entities_response[:5]: # Print ordered upto n rows
    print(f"Key={entity.key}\tEntity={entity.display_name}\tSource={entity.external_key}") if ".pdb" in entity.display_name.lower() else None

Key=a7e47428-6ae3-464e-bdac-907b78375484	Entity=data/design-info.json.1	Source=https://swiftobjectstorage.us-chicago-1.oraclecloud.com/v1/orasenatdpltintegration03/Ellison/data/design-info.json.1
Key=2d54f783-c3f9-4528-a007-78ab520f2ccf	Entity=data/design-info.json.10	Source=https://swiftobjectstorage.us-chicago-1.oraclecloud.com/v1/orasenatdpltintegration03/Ellison/data/design-info.json.10


In [76]:
# ----------------------------------------------------------------------
# List all Glossaries defined in the data catalog
# ----------------------------------------------------------------------
# Initialize service client with default config file
list_glossaries_response = data_catalog_client.list_glossaries(
    catalog_id=ids)

# Get the data from response
#print(list_glossaries_response.data)
for glossary in list_glossaries_response.data.items:
    print(f"Key={glossary.key}\tGlossary={glossary.display_name}") if "ellison" in glossary.display_name.lower() else None

Key=705e4631-6787-4073-9efe-9b2b1d5a4bb4	Glossary=Ellison-Business-Glossary


In [77]:
# ----------------------------------------------------------------------
# List all namespaces for the custom properties
# ----------------------------------------------------------------------
list_namespaces_response = data_catalog_client.list_namespaces(
    catalog_id=ids,)

# Get the data from response
#print(list_namespaces_response.data)
for namespace in list_namespaces_response.data.items:
    print(f"Key={namespace.key}\tNamespace={namespace.display_name}")

Key=0e4d60d9-d5b5-467f-89bb-22db63a3ee18	Namespace=Custom Properties


#### Updating custom properties for an entity

In [78]:
# ----------------------------------------------------------------------
# List all custom properties for PSFT for the namespace
# ----------------------------------------------------------------------
NAMESPACE_ID="0e4d60d9-d5b5-467f-89bb-22db63a3ee18" # from above
list_custom_properties_response = data_catalog_client.list_custom_properties(
    catalog_id=ids,
    namespace_id=NAMESPACE_ID,)

# Get the data from response
#print(list_custom_properties_response.data)
for cp in list_custom_properties_response.data.items:
    print(f"Namespace={NAMESPACE_ID}\tKey={cp.key}\tCustom property={cp.display_name}") if not ("psft" in cp.display_name.lower()) else None

Namespace=0e4d60d9-d5b5-467f-89bb-22db63a3ee18	Key=80d20f60-c922-4f35-ae0a-f057df4b8a3e	Custom property=COUNT_AMINO_ACIDS_Y
Namespace=0e4d60d9-d5b5-467f-89bb-22db63a3ee18	Key=ef66923c-e410-41d8-a9cb-c659cfa7725c	Custom property=SCI_FUNC
Namespace=0e4d60d9-d5b5-467f-89bb-22db63a3ee18	Key=d9c4d6cf-3aa7-4dc1-9a86-c8c201b12f94	Custom property=FILE_TYPE
Namespace=0e4d60d9-d5b5-467f-89bb-22db63a3ee18	Key=7e03b6da-f425-4af2-9f0d-547c8cdcb02a	Custom property=AROMATICITY
Namespace=0e4d60d9-d5b5-467f-89bb-22db63a3ee18	Key=ea45540c-9d68-4067-af53-03d373283542	Custom property=AMINO_ACIDS_PERCENT_A
Namespace=0e4d60d9-d5b5-467f-89bb-22db63a3ee18	Key=c832f731-6505-4b4b-82e3-044bb95a00b0	Custom property=COUNT_AMINO_ACIDS_W


In [79]:
# ----------------------------------------------------------------------
# List allowed values for a list of custom properties
# ----------------------------------------------------------------------
NAMESPACE_ID="0e4d60d9-d5b5-467f-89bb-22db63a3ee18"
cp_keys = {cp.key: None for cp in list_custom_properties_response.data.items}

for cp_key in cp_keys:
    response = data_catalog_client.get_custom_property(
    	catalog_id=ids,
    	namespace_id=NAMESPACE_ID,
    	custom_property_key=cp_key,
    	)
    # Get the data from response
    data = response.data
    print(f"cp_key={cp_key}\tName={data.display_name}\tAllowed values={data.allowed_values}") if not ("psft" in data.display_name.lower()) else None

cp_key=80d20f60-c922-4f35-ae0a-f057df4b8a3e	Name=COUNT_AMINO_ACIDS_Y	Allowed values=["1", "2", "3", "4", "5", "6", "8"]
cp_key=ef66923c-e410-41d8-a9cb-c659cfa7725c	Name=SCI_FUNC	Allowed values=["mhc", "scores", "design"]


cp_key=d9c4d6cf-3aa7-4dc1-9a86-c8c201b12f94	Name=FILE_TYPE	Allowed values=["json", "pdb", "fa"]
cp_key=7e03b6da-f425-4af2-9f0d-547c8cdcb02a	Name=AROMATICITY	Allowed values=[".02", ".03", "0.05", ".06", ".08", ".09", ".11", ".12", ".14"]
cp_key=ea45540c-9d68-4067-af53-03d373283542	Name=AMINO_ACIDS_PERCENT_A	Allowed values=[".030769231", ".015384615", ".046153846", ".061538462", ".076923077", ".092307692", ".1076923076923077", ".123076923", ".1384615384615384", ".1538461538461538", ".1846153846153846", ".2", "0"]


cp_key=c832f731-6505-4b4b-82e3-044bb95a00b0	Name=COUNT_AMINO_ACIDS_W	Allowed values=["0", "1", "2", "3"]


In [86]:
# ----------------------------------------------------------------------
# Getting custom properties LOV and assigned values for an Entity
# ----------------------------------------------------------------------
ASSET_KEY="672dd4d1-04d8-4839-9016-23654bb6c41e"
ENTITY_KEY="a7e47428-6ae3-464e-bdac-907b78375484"
get_entity_response = data_catalog_client.get_entity(
    catalog_id=ids,
    data_asset_key=ASSET_KEY,
    entity_key=ENTITY_KEY,
   )
entity_name=(get_entity_response.data.display_name)
#print(get_entity_response.data.custom_property_members)
for cp in get_entity_response.data.custom_property_members:
    print(f"Entity={entity_name}\tname={cp.display_name}\tallowed_values={cp.allowed_values}\tAssignedcpkey={cp.key}\tAssigned={cp.value}")


Entity=data/design-info.json.1	name=SCI_FUNC	allowed_values=["mhc", "scores", "design"]	Assignedcpkey=None	Assigned=None
Entity=data/design-info.json.1	name=FILE_TYPE	allowed_values=["json", "pdb", "fa"]	Assignedcpkey=None	Assigned=None


In [81]:
# ----------------------------------------------------------------------
# Updating custom properties assigned values for an Entity
# ----------------------------------------------------------------------
NEW_FILE_TYPE="json"
NEW_SCI_FUNC="design"
ASSET_KEY="672dd4d1-04d8-4839-9016-23654bb6c41e"
ENTITY_KEY="a7e47428-6ae3-464e-bdac-907b78375484"
update_entity_response = data_catalog_client.update_entity(
    catalog_id=ids,
    data_asset_key=ASSET_KEY,
    entity_key=ENTITY_KEY,
    update_entity_details=oci.data_catalog.models.UpdateEntityDetails(
        custom_property_members=[
            oci.data_catalog.models.CustomPropertySetUsage(
                    key="d9c4d6cf-3aa7-4dc1-9a86-c8c201b12f94", #FILE_TYPE key
                    display_name="FILE_TYPE",
                    value=NEW_FILE_TYPE,
                    namespace_name="Custom Properties"
                    ),
            oci.data_catalog.models.CustomPropertySetUsage(
                    key="ef66923c-e410-41d8-a9cb-c659cfa7725c", #SCI_FUNC_KEY
                    display_name="SCI_FUNC",
                    value=NEW_SCI_FUNC,
                    namespace_name="Custom Properties"
                    )],)
        )

Command ID e13b9052-ee89-4d1e-983b-64b5960aa261 failed with java.lang.RuntimeException: [Command e13b9052-ee89-4d1e-983b-64b5960aa261 in Context f2f8950b-ea4e-4029-a84d-b6f6f00e733f] failed with error: 
 ---------------------------------------------------------------------------ServiceError                              Traceback (most recent call last)Cell In[289], line 8
      6 ASSET_KEY="672dd4d1-04d8-4839-9016-23654bb6c41e"
      7 ENTITY_KEY="a7e47428-6ae3-464e-bdac-907b78375484"
----> 8 update_entity_response = data_catalog_client.update_entity(
      9     catalog_id=ids,
     10     data_asset_key=ASSET_KEY,
     11     entity_key=ENTITY_KEY,
     12     update_entity_details=oci.data_catalog.models.UpdateEntityDetails(
     13         custom_property_members=[
     14             oci.data_catalog.models.CustomPropertySetUsage(
     15                     key="d9c4d6cf-3aa7-4dc1-9a86-c8c201b12f94", #FILE_TYPE key
     16                     display_name="FILE_TYPE",
     17    

In [ ]:
# Get the data from response
#print(update_entity_response.data)

#### Checking out harvest jobs and incremental executions

In [92]:
# ----------------------------------------------------------------------
# List all job definitions for the PSFT data asset
# ----------------------------------------------------------------------
ASSET_KEY="672dd4d1-04d8-4839-9016-23654bb6c41e" # Ellison OS jobs
DISPLAY_CONTAINS="Harvest"
list_job_definitions_response = data_catalog_client.list_job_definitions(
    catalog_id=ids,
    data_asset_key=ASSET_KEY,
    display_name_contains=DISPLAY_CONTAINS)

# Get the data from response
#print(list_job_definitions_response.data)
for jobdef in list_job_definitions_response.data.items:
    print(f"Key={jobdef.key}\tName={jobdef.display_name}\tJob type={jobdef.job_type}\tStatus={jobdef.job_execution_state}\tlast execution={jobdef.time_latest_execution_ended}")

Key=d878f217-6cf1-4bd8-9ad5-befa2c2335db	Name=Harvest_Ellison_OS_20260505122429	Job type=HARVEST	Status=SUCCEEDED	last execution=2026-05-05 17:25:23.358878+00:00
Key=18b27367-a685-4581-9cd4-478cf465c21e	Name=Harvest_Ellison_OS_20260505124630	Job type=HARVEST	Status=SUCCEEDED	last execution=2026-05-05 17:47:35.421651+00:00


In [93]:
# ----------------------------------------------------------------------
# List jobs
# ----------------------------------------------------------------------
list_jobs_response = data_catalog_client.list_jobs(
    catalog_id=ids,
    data_asset_key=ASSET_KEY,
    display_name_contains=DISPLAY_CONTAINS)

# Get the data from response
#print(list_jobs_response.data)
for job in list_jobs_response.data.items:
    print(f"Key={job.key}\tName={job.display_name}\tStatus={lifecycle_state}\tExec Count={job.execution_count}\tStatus={job.time_of_latest_execution}")

Key=179947f8-2280-49da-a1ec-4ffee108250d	Name=Harvest_Ellison_OS_20260505122429_job	Status=ACTIVE	Exec Count=1	Status=2026-05-05 17:24:38.890781+00:00
Key=ea51f86d-f27e-4e1f-bbfd-3afb0096cf91	Name=Harvest_Ellison_OS_20260505124630_job	Status=ACTIVE	Exec Count=1	Status=2026-05-05 17:46:50.978833+00:00


In [94]:
JOB_KEY="179947f8-2280-49da-a1ec-4ffee108250d"
JOB_TYPE="HARVEST"
list_job_executions_response = data_catalog_client.list_job_executions(
    catalog_id=ids,
    job_key=JOB_KEY,
    job_type=JOB_TYPE)

# Get the data from response
#print(list_job_executions_response.data)
for jobexec in list_job_executions_response.data.items:
    print(f"Job Key={jobexec.job_key}\tJob Exec key={jobexec.key}\tType={jobexec.job_type}\tStatus={lifecycle_state}\time ended={jobexec.time_ended}")

Job Key=179947f8-2280-49da-a1ec-4ffee108250d	Job Exec key=d6912283-3dcc-42eb-b227-d8b60dec5542	Type=HARVEST	Status=ACTIVE	ime ended=2026-05-05 17:25:23.358878+00:00


In [90]:
# ----------------------------------------------------------------------
# Execute an existing harvest job 
# ----------------------------------------------------------------------
create_job_execution_response = data_catalog_client.create_job_execution(
    catalog_id=ids,
    job_key=JOB_KEY,
    create_job_execution_details=oci.data_catalog.models.CreateJobExecutionDetails(
        job_type=JOB_TYPE,
        ),
        )

# Get the data from response
job=create_job_execution_response.data
print(f"Job Key={job.job_key}\tJob Exec key={job.key}\tType={job.job_type}\tStatus={job.lifecycle_state}")

Job Key=d50e725f-a34d-414f-bcad-bf692ce4d62a	Job Exec key=10932a66-dff7-45d9-af94-74bd5d494b77	Type=HARVEST	Status=CREATED
